In [3]:
import pandas as pd
import requests
import os

# --------------------------------------------------------
# 1. FUENTE 1: DATOS LOCALES (Simulamos un CSV para reproducibilidad)
# --------------------------------------------------------
os.makedirs('../data', exist_ok=True)
datos_mock = pd.DataFrame({
    'id_envio': [101, 102, 103, 104, 105],
    'cp_origen': ['55600', '55600', '55600', '55600', '55600'], 
    'cp_destino': ['01000', '64000', '44100', '97000', '01000'],
    'paqueteria': ['FedEx', 'DHL', 'Estafeta', 'FedEx', 'DHL'],
    'costo_mxn': [850, 1200, 980, 2300, 800],
    'tiempo_dias': [1, 2, 2, 4, 1]
})
datos_mock.to_csv('../data/envios_historicos.csv', index=False)

df_envios = pd.read_csv('../data/envios_historicos.csv')

print("--- EXTRACCIÓN: Histórico de Envíos ---")
display(df_envios.head())
print("Dimensiones:", df_envios.shape)
print("Tipos de datos:\n", df_envios.dtypes)

# --------------------------------------------------------
# 2. FUENTE 2: EXTRACCIÓN AUTOMATIZADA (API)
# --------------------------------------------------------
def obtener_estado_por_cp(cp):
    """Consulta la API pública para obtener el estado de un CP en México"""
    url = f"https://api.zippopotam.us/mx/{cp}"
    respuesta = requests.get(url)
    if respuesta.status_code == 200:
        return respuesta.json()['places'][0]['state']
    return "Desconocido"

# Aplicamos la API al dataframe
df_envios['estado_destino'] = df_envios['cp_destino'].astype(str).str.zfill(5).apply(obtener_estado_por_cp)

print("\n--- EXTRACCIÓN API: Datos Geográficos ---")
display(df_envios[['cp_destino', 'estado_destino']].head())

# --------------------------------------------------------
# 3. DIAGNÓSTICO INICIAL
# --------------------------------------------------------
print("\n--- DIAGNÓSTICO INICIAL ---")
print("Valores nulos por columna:\n", df_envios.isnull().sum())
print("Filas duplicadas en la base:", df_envios.duplicated().sum())
print("Limitaciones: La API tiene un límite de consultas por minuto, por lo que para bases masivas reales se requerirá procesamiento asíncrono o una base de datos de CPs.")

# --------------------------------------------------------
# 4. PRIMERA EVIDENCIA
# --------------------------------------------------------
print("\n--- PRIMERA EVIDENCIA ---")
print("Pregunta: ¿Cuál es el costo promedio de flete por paquetería según nuestra muestra?")
evidencia = df_envios.groupby('paqueteria')[['costo_mxn', 'tiempo_dias']].mean().reset_index()
display(evidencia)
print("Interpretación: Con esta muestra preliminar, DHL y FedEx presentan los costos promedio más altos, sin embargo, FedEx absorbió el viaje más largo (CP 97000 - Yucatán), lo que infla su promedio.")

--- EXTRACCIÓN: Histórico de Envíos ---


,id_envio,cp_origen,cp_destino,paqueteria,costo_mxn,tiempo_dias
0,101,55600,1000,FedEx,850,1
1,102,55600,64000,DHL,1200,2
2,103,55600,44100,Estafeta,980,2
3,104,55600,97000,FedEx,2300,4
4,105,55600,1000,DHL,800,1


Dimensiones: (5, 6)
Tipos de datos:
 id_envio        int64
cp_origen       int64
cp_destino      int64
paqueteria     object
costo_mxn       int64
tiempo_dias     int64
dtype: object

--- EXTRACCIÓN API: Datos Geográficos ---


,cp_destino,estado_destino
0,1000,Distrito Federal
1,64000,Nuevo Leon
2,44100,Jalisco
3,97000,Yucatan
4,1000,Distrito Federal



--- DIAGNÓSTICO INICIAL ---
Valores nulos por columna:
 id_envio          0
cp_origen         0
cp_destino        0
paqueteria        0
costo_mxn         0
tiempo_dias       0
estado_destino    0
dtype: int64
Filas duplicadas en la base: 0
Limitaciones: La API tiene un límite de consultas por minuto, por lo que para bases masivas reales se requerirá procesamiento asíncrono o una base de datos de CPs.

--- PRIMERA EVIDENCIA ---
Pregunta: ¿Cuál es el costo promedio de flete por paquetería según nuestra muestra?


,paqueteria,costo_mxn,tiempo_dias
0,DHL,1000.0,1.5
1,Estafeta,980.0,2.0
2,FedEx,1575.0,2.5


Interpretación: Con esta muestra preliminar, DHL y FedEx presentan los costos promedio más altos, sin embargo, FedEx absorbió el viaje más largo (CP 97000 - Yucatán), lo que infla su promedio.
